# WordNet / VerbNet / FrameNet 动手看

每节开头有一个变量（`NOUN`、`VERB`、`ADJ`……），改掉它重跑就能换词。
尽量打印原始对象，解释留给你自己看。最后一节和 `wn_candidates.candidates()` 对照。

环境：`repairsbn`（NLTK 3.10，三个语料都已下载）。

In [1]:
from nltk.corpus import wordnet as wn
from nltk.corpus import verbnet as vn
from nltk.corpus import framenet as fn

## 1. WordNet

WordNet 的基本单位是 **synset**（一组同义的词义），不是词。一个词形对应多个 synset，
一个 synset 包含多个 **lemma**（写法）。SBN 里的 `girl.n.01` 就是一个 synset id。

### 1.1 一个词形有哪些 synset

In [2]:
NOUN = "girl"
for s in wn.synsets(NOUN):
    print(s.name(), "|", s.pos(), "|", s.lexname(), "|", s.definition())

girl.n.01 | n | noun.person | a young woman
female_child.n.01 | n | noun.person | a youthful female person
daughter.n.01 | n | noun.person | a female human offspring
girlfriend.n.02 | n | noun.person | a girl or young woman with whom a man is romantically involved
girl.n.05 | n | noun.person | a friendly informal reference to a grown woman


### 1.2 一个 synset 里面有什么

In [3]:
ss = wn.synset("girl.n.01")
print("definition:", ss.definition())
print("examples:  ", ss.examples())
print("lemmas:    ", ss.lemmas())
print("lemma names", ss.lemma_names())
print("lexname:   ", ss.lexname())   # 语义大类（noun.person 等）

definition: a young woman
examples:   ['a young lady of 18']
lemmas:     [Lemma('girl.n.01.girl'), Lemma('girl.n.01.miss'), Lemma('girl.n.01.missy'), Lemma('girl.n.01.young_lady'), Lemma('girl.n.01.young_woman'), Lemma('girl.n.01.fille')]
lemma names ['girl', 'miss', 'missy', 'young_lady', 'young_woman', 'fille']
lexname:    noun.person


### 1.3 lemma 层：sense key 和计数

`sense key` 是 VerbNet 连回 WordNet 用的 ID（第 2 节会用到）。
`count()` 是 SemCor 里的频次，`_commonness` 用的是 synset 编号和 lemma 数，不是这个。

In [4]:
for l in ss.lemmas():
    print(l.name(), "| key:", l.key(), "| count:", l.count())

girl | key: girl%1:18:00:: | count: 80
miss | key: miss%1:18:00:: | count: 3
missy | key: missy%1:18:00:: | count: 0
young_lady | key: young_lady%1:18:00:: | count: 1
young_woman | key: young_woman%1:18:00:: | count: 4
fille | key: fille%1:18:00:: | count: 0


### 1.4 名词层级：上位词链、下义词、同位下义词（co-hyponyms）

In [5]:
for path in ss.hypernym_paths():
    print(" -> ".join(s.name() for s in path))

entity.n.01 -> physical_entity.n.01 -> object.n.01 -> whole.n.02 -> living_thing.n.01 -> organism.n.01 -> person.n.01 -> adult.n.01 -> woman.n.01 -> girl.n.01
entity.n.01 -> physical_entity.n.01 -> causal_agent.n.01 -> person.n.01 -> adult.n.01 -> woman.n.01 -> girl.n.01
entity.n.01 -> physical_entity.n.01 -> object.n.01 -> whole.n.02 -> living_thing.n.01 -> organism.n.01 -> person.n.01 -> female.n.02 -> woman.n.01 -> girl.n.01
entity.n.01 -> physical_entity.n.01 -> causal_agent.n.01 -> person.n.01 -> female.n.02 -> woman.n.01 -> girl.n.01


In [6]:
print("hypernyms:", ss.hypernyms())
print("hyponyms: ", ss.hyponyms()[:10], "...")

hypernyms: [Synset('woman.n.01')]
hyponyms:  [Synset('mill-girl.n.01'), Synset('tomboy.n.01'), Synset('lass.n.01'), Synset('dame.n.01'), Synset('rosebud.n.02'), Synset('working_girl.n.01'), Synset('sweater_girl.n.01'), Synset('gamine.n.02'), Synset('valley_girl.n.01'), Synset('soubrette.n.01')] ...


In [7]:
# wn_candidates._cohyponyms 就是这两行：上位词的所有下义词，去掉自己
co = {h for hyper in ss.hypernyms() for h in hyper.hyponyms()} - {ss}
print(len(co))
sorted(s.name() for s in co)

57


['amazon.n.01',
 'b-girl.n.01',
 'bachelor_girl.n.01',
 'baggage.n.02',
 'ball-buster.n.01',
 'black_woman.n.01',
 'bluestocking.n.01',
 'bridesmaid.n.01',
 'broad.n.01',
 'cat.n.03',
 'cinderella.n.01',
 'coquette.n.01',
 'dame.n.02',
 'debutante.n.01',
 'divorcee.n.01',
 'dominatrix.n.01',
 'donna.n.01',
 'enchantress.n.01',
 'ex-wife.n.01',
 'eyeful.n.01',
 'geisha.n.01',
 'girl.n.05',
 'girlfriend.n.01',
 'girlfriend.n.02',
 'gold_digger.n.02',
 'gravida.n.02',
 'heroine.n.02',
 'inamorata.n.01',
 'jezebel.n.02',
 'jilt.n.01',
 'lady.n.01',
 'maenad.n.01',
 'maenad.n.02',
 'matriarch.n.01',
 'matriarch.n.02',
 'matron.n.03',
 'mestiza.n.01',
 'mistress.n.01',
 'mother_figure.n.01',
 'nanny.n.01',
 'nullipara.n.01',
 'nymph.n.03',
 'nymphet.n.01',
 'old_woman.n.01',
 'prostitute.n.01',
 'shiksa.n.01',
 'smasher.n.02',
 'sylph.n.01',
 'unmarried_woman.n.01',
 'vestal.n.01',
 'wac.n.01',
 'wave.n.09',
 'white_woman.n.01',
 'widow.n.01',
 'wife.n.01',
 'wonder_woman.n.01',
 'yellow_wom

### 1.5 动词：层级很浅，verb groups 是近义

`wn_candidates` 不用 verb groups，就是因为 WordNet 把它定义为同一动词的近义词义。

In [8]:
VERB = "eat"
for s in wn.synsets(VERB, pos="v"):
    print(s.name(), "|", s.definition())

eat.v.01 | take in solid food
eat.v.02 | eat a meal; take a meal
feed.v.06 | take in food; used of animals only
eat.v.04 | worry or cause anxiety in a persistent way
consume.v.05 | use up (resources or materials)
corrode.v.01 | cause to deteriorate due to the action of water, air, or an acid


In [9]:
vs = wn.synset("eat.v.01")
for path in vs.hypernym_paths():
    print(" -> ".join(s.name() for s in path))
print("verb_groups:", vs.verb_groups())
print("entailments:", vs.entailments())

consume.v.02 -> eat.v.02 -> eat.v.01
consume.v.02 -> eat.v.01
verb_groups: [Synset('feed.v.06'), Synset('eat.v.02')]
entailments: [Synset('chew.v.01'), Synset('swallow.v.01')]


### 1.6 形容词：head（`.a.`）和 satellite（`.s.`）

SBN 只写得出 `.a.`。看一下 `similar_tos()` 返回的全是 `.s.`，`also_sees()` 返回的是近义的 head。
反义关系在 **lemma** 上，不在 synset 上。

In [10]:
ADJ = "new"
for s in wn.synsets(ADJ, pos="a"):   # pos="a" 同时返回 a 和 s
    print(s.name(), "|", s.pos(), "|", s.definition())

new.a.01 | a | not of long duration; having just (or relatively recently) come into being or been made or acquired or discovered
fresh.s.03 | s | original and of a kind not seen before
raw.s.11 | s | lacking training or experience
new.s.03 | s | having no previous example or precedent or parallel
new.s.04 | s | other than the former one(s); different
new.a.06 | a | unaffected by use or exposure
newfangled.s.01 | s | (of a new kind or fashion) gratuitously new
new.s.06 | s | in use after medieval times
modern.s.04 | s | used of a living language; being the current stage in its development
new.s.08 | s | (of crops) harvested at an early stage of development; before complete maturity
new.s.09 | s | unfamiliar


In [11]:
a = wn.synset("new.a.01")
print("similar_tos:", a.similar_tos())
print("also_sees:  ", a.also_sees())
print("antonyms:   ", [(l.name(), l.antonyms()) for l in a.lemmas()])

similar_tos: [Synset('hot.s.11'), Synset('brand-new.s.01'), Synset('sunrise.s.01'), Synset('unused.s.02'), Synset('parvenu.s.02'), Synset('fresh.s.01'), Synset('recent.s.01'), Synset('newborn.s.02'), Synset('revolutionary.s.01'), Synset('young.s.03'), Synset('untested.s.02'), Synset('novel.s.02'), Synset('rising.s.03'), Synset('virgin.s.01'), Synset('newfound.s.01')]
also_sees:   [Synset('modern.a.01'), Synset('fresh.a.01'), Synset('current.a.01')]
antonyms:    [('new', [Lemma('old.a.02.old')])]


## 2. VerbNet

VerbNet 的单位是 **class**（`eat-39.1`），按共享的句法框架把动词分组。
class 可以有子类（`eat-39.1-1`）。每个 member 通过 WordNet sense key 连回 WordNet。

### 2.1 一个动词在哪些 class 里

In [12]:
VERB = "eat"
vn.classids(lemma=VERB)

['eat-39.1-1']

### 2.2 一个 class 的全貌（NLTK 自带的漂亮打印）

In [13]:
CID = vn.classids(lemma=VERB)[0]
print(vn.pprint(CID))

eat-39.1-1
  Subclasses: (none)
  Members: eat
  Thematic roles:
    * Patient[+comestible +solid]
  Frames:
    Basic Transitive
      Example: Cynthia ate the peach.
      Syntax: NP[Agent] VERB NP[Patient]
      Semantics:
        * take_in(during(E), Agent, Patient)
    Unspecified Object
      Example: Cynthia ate.
      Syntax: NP[Agent] VERB
      Semantics:
        * take_in(during(E), Agent, ?Patient)
    Conative
      Example: Cynthia ate at the peach.
      Syntax: NP[Agent] VERB PREP[at] NP[Patient]
      Semantics:
        * take_in(during(E), Agent, Patient)
    NP-ADJP (Resultative)
      Example: Cynthia ate herself sick.
      Syntax: NP[Agent] VERB NP[Oblique +refl] ADJ
      Semantics:
        * take_in(during(E), Agent, ?Patient)
        * Pred(result(E), Oblique)
    PP (Source-PP)
      Example: He ate off of the table.
      Syntax: NP[Agent] VERB PREP[+src] NP[Source]
      Semantics:
        * take_in(during(E), Agent, ?Patient)
        * Prep(during(E), ?Pati

### 2.3 底层是 XML

`vn.vnclass()` 返回的是 ElementTree 元素。`wn_candidates` 直接读 `MEMBERS/MEMBER` 的 `wn` 属性。

In [14]:
import xml.etree.ElementTree as ET
el = vn.vnclass(CID)
print(ET.tostring(el, encoding="unicode")[:2000])

<VNSUBCLASS ID="eat-39.1-1">
            <MEMBERS>
                <MEMBER name="eat" wn="eat%2:34:00 eat%2:34:01 eat%2:34:02" />
            </MEMBERS>
            <THEMROLES>
                <THEMROLE type="Patient">
                    <SELRESTRS>
                        <SELRESTR Value="+" type="comestible" />
                        <SELRESTR Value="+" type="solid" />
                    </SELRESTRS>
                </THEMROLE>
            </THEMROLES>
            <FRAMES>
                <FRAME>
                    <DESCRIPTION descriptionNumber="" primary="Basic Transitive" secondary="" xtag="" />
                    <EXAMPLES>
                        <EXAMPLE>Cynthia ate the peach.</EXAMPLE>
                    </EXAMPLES>
                    <SYNTAX>
                        <NP value="Agent">
                            <SYNRESTRS />
                        </NP>
                        <VERB />
                        <NP value="Patient">
                            <SYNRESTR

In [15]:
for m in el.findall("MEMBERS/MEMBER"):
    print(m.attrib)

{'name': 'eat', 'wn': 'eat%2:34:00 eat%2:34:01 eat%2:34:02'}


### 2.4 父类与子类

`wn_candidates._verbnet_index` 把 class 截到前两段（`eat-39.1-1` → `eat-39.1`），
让单成员的子类也能拿到兄弟。这里看看父类和兄弟类里有什么。

In [16]:
parent = "-".join(CID.split("-")[:2])
print("parent:", parent)
print("subclasses:", vn.subclasses(parent))
for c in [parent] + vn.subclasses(parent):
    print(c, "->", vn.lemmas(c))

parent: eat-39.1
subclasses: ['eat-39.1-1', 'eat-39.1-2']
eat-39.1 -> []
eat-39.1-1 -> ['eat']
eat-39.1-2 -> ['drink']


### 2.5 从 sense key 回到 WordNet synset

注意父类 `eat-39.1` 自己没有 member，内容都在子类上，所以这里遍历子类。
VerbNet 的 key 少了结尾的 `::`，NLTK 要补上才能查（`_norm_key` 做的就是这个）。

In [17]:
for c in vn.subclasses(parent) or [parent]:
    for key in vn.wordnetids(c):
        try:
            print(c, key, "->", wn.lemma_from_key(key + "::").synset())
        except Exception as e:
            print(c, key, "-> (lookup failed:", type(e).__name__, ")")

eat-39.1-1 eat%2:34:00 -> Synset('eat.v.01')
eat-39.1-1 eat%2:34:01 -> Synset('eat.v.02')
eat-39.1-1 eat%2:34:02 -> Synset('feed.v.06')
eat-39.1-2 drink%2:34:00 -> Synset('drink.v.01')
eat-39.1-2 drink%2:34:01 -> Synset('drink.v.02')
eat-39.1-2 drink%2:34:12 -> Synset('toast.v.02')


### 2.6 句法框架：为什么说 class-mates 共享论元结构

In [18]:
for c in vn.subclasses(parent) or [parent]:
    print("==", c)
    for fr in vn.frames(c):
        print("  ", fr["description"]["primary"], "|", fr["example"])

== eat-39.1-1
   Basic Transitive | Cynthia ate the peach.
   Unspecified Object | Cynthia ate.
   Conative | Cynthia ate at the peach.
   NP-ADJP | Cynthia ate herself sick.
   PP | He ate off of the table.
== eat-39.1-2
   Basic Transitive | Cynthia drank the wine.
   Unspecified Object | Cynthia drank.
   NP-ADJP | Cythia drank herself sick.
   PP | He drank out of the goblet.


## 3. FrameNet

FrameNet 的单位是 **frame**（一个场景或属性维度，如 `Chemical-sense_description`）。
frame 有 **frame elements**（FE，场景的参与者/槽位）和 **lexical units**（LU，能唤起这个 frame 的词，形如 `spicy.a`）。
形容词的 frame 往往就是一个属性维度，同 frame 的形容词 LU 就是这个维度上的不同取值。

### 3.1 一个形容词对应哪些 LU、哪些 frame

In [19]:
ADJ = "spicy"
lus = fn.lus(rf"^{ADJ}\.a$")
for lu in lus:
    print(lu.ID, lu.name, "->", lu.frame.name)

4943 spicy.a -> Chemical-sense_description


### 3.2 LU 对象本身

In [20]:
lu = lus[0]
print(lu)

lexical unit (4943): spicy.a

[definition]
  COD: flavoured with spice.
[frame] Chemical-sense_description(271)

[POS] A

[status] Finished_Initial

[lexemes] spicy/A

[semTypes] 0 semantic types

[URL] https://framenet2.icsi.berkeley.edu/fnReports/data/lu/lu4943.xml

[subCorpus] 8 subcorpora
  A-002-s20-rcoll-dish, A-002-s20-rcoll-flavour,
  A-002-s20-rcoll-food, A-002-s20-rcoll-sauce,
  A-002-s20-rcoll-sausage, A-170-s20-prenom, A-180-s20-punct,
  A-190-s20-other

[exemplars] 17 sentences across all subcorpora



### 3.3 frame 对象：定义、FE、LU

In [21]:
frame = lu.frame
print(frame.definition[:600])

This  frame covers the descriptions of tastes and smells, and has one main frame element, Source. The frame element Sensory_attribute occurs mainly with the words taste, flavour, and smell.  'The almonds had a bitter taste.' 'The almonds are bitter.'


In [22]:
for name, fe in frame.FE.items():
    print(f"{name:25s} {fe.coreType:12s} {fe.definition[:80]!r}")

Perceptual_source         Core         'Perceptual_source is used for the entity that is being described. A recipe can s'
Degree                    Peripheral   "This FE identifies the Degree to which a chemical sense is described.  'The almo"
Sensory_attribute         Core         "Sensory_attribute is used for words that describe the perception itself. \t 'The "


In [23]:
# 这个 frame 里所有 LU，按词性分组
from collections import defaultdict
by_pos = defaultdict(list)
for name in frame.lexUnit:
    lem, pos = name.rsplit(".", 1)
    by_pos[pos].append(lem)
dict(by_pos)

{'a': ['tasty',
  'piquant',
  'yummy',
  'flavourful',
  'scrumptious',
  'palatable',
  'delectable',
  'flavoursome',
  'toothsome',
  'ambrosial',
  'sapid',
  'salty',
  'sour',
  'bitter',
  'spicy',
  'sweet',
  'hot',
  'savory',
  'delicious',
  'pungent',
  'tart',
  'flavourless',
  'bland',
  'insipid',
  'unpalatable',
  'smelly',
  'aromatic',
  'fragrant',
  'tasteless',
  'malodorous'],
 'n': ['stench',
  'odor',
  'reek',
  'stink',
  'aroma',
  'fragrance',
  'scent',
  'bouquet'],
 'v': ['reek', 'stink', 'smell']}

### 3.4 frame 之间的关系（继承、使用……）

In [24]:
for r in frame.frameRelations:
    print(r)

<Parent=Gradable_attributes -- Inheritance -> Child=Chemical-sense_description>
<Parent=Give_impression -- Using -> Child=Chemical-sense_description>


### 3.5 FrameNet 只给 lemma，SBN 要 synset：`_frame_mates` 取第一个 `.a.` 词义

In [25]:
for lem in sorted(by_pos["a"]):
    heads = [s.name() for s in wn.synsets(lem.replace(" ", "_"), pos="a") if s.pos() == "a"]
    print(f"{lem:15s} -> {heads[:3]}")

ambrosial       -> []
aromatic        -> ['aromatic.a.01']
bitter          -> []
bland           -> []
delectable      -> []
delicious       -> []
flavourful      -> []
flavourless     -> []
flavoursome     -> []
fragrant        -> ['fragrant.a.01']
hot             -> ['hot.a.01', 'hot.a.03']
insipid         -> []
malodorous      -> ['malodorous.a.01']
palatable       -> ['palatable.a.01']
piquant         -> []
pungent         -> []
salty           -> ['salty.a.02']
sapid           -> []
savory          -> ['savory.a.01']
scrumptious     -> []
smelly          -> []
sour            -> ['sour.a.02']
spicy           -> []
sweet           -> ['sweet.a.01', 'sweet.a.07', 'fresh.a.06']
tart            -> []
tasteless       -> ['tasteless.a.01', 'tasteless.a.02']
tasty           -> ['tasty.a.01']
toothsome       -> ['palatable.a.01']
unpalatable     -> ['unpalatable.a.01']
yummy           -> []


空列表不是查询失败，是这个词在 WordNet 里**只有 satellite 词义**，SBN 写不出来，于是从池里掉了。
下面把它们的全部词义打出来看。

In [26]:
for lem in sorted(by_pos["a"]):
    ss_all = wn.synsets(lem.replace(" ", "_"), pos="a")
    if not any(s.pos() == "a" for s in ss_all):
        print(f"{lem:15s}", [s.name() for s in ss_all] or "(not in WordNet)")

ambrosial       ['ambrosial.s.01', 'ambrosial.s.02']
bitter          ['acrimonious.s.01', 'bitter.s.02', 'acerb.s.02', 'bitter.s.04', 'bitter.s.05', 'bitter.s.06', 'biting.s.02']
bland           ['bland.s.01', 'bland.s.02', 'politic.s.01']
delectable      ['delectable.s.01', 'delectable.s.02']
delicious       ['delightful.s.01', 'delectable.s.01']
flavourful      ['flavorful.s.01']
flavourless     ['bland.s.01']
flavoursome     ['flavorful.s.01']
insipid         ['bland.s.01', 'insipid.s.02']
piquant         ['piquant.s.01', 'piquant.s.02', 'engaging.s.01']
pungent         ['pungent.s.01', 'barbed.s.01']
sapid           ['flavorful.s.01']
scrumptious     ['delectable.s.01']
smelly          ['fetid.s.01']
spicy           ['piquant.s.01', 'hot.s.07', 'blue.s.05']
tart            ['lemony.s.01', 'sharp.s.05']
yummy           ['delectable.s.01']


## 4. 对照 `wn_candidates.candidates()`

上面手工看到的东西，和生成器实际拿到的候选池比一下。
`candidates()` 做了三件事：按词性取池、去掉写不成 SBN 的（`.s.`）、去掉共享 lemma 的。

In [27]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))   # 在 repair_data_gen/ 下启动 notebook
import wn_candidates as wc

for sid in ["girl.n.01", "eat.v.01", "spicy.a.01", "new.a.01"]:
    print(sid, "->", wc.candidates(sid))

girl.n.01 -> ('prostitute.n.01', 'coquette.n.01', 'enchantress.n.01', 'mistress.n.01', 'nanny.n.01', 'amazon.n.01', 'b-girl.n.01', 'ball-buster.n.01', 'bluestocking.n.01', 'bridesmaid.n.01', 'debutante.n.01', 'divorcee.n.01', 'ex-wife.n.01', 'geisha.n.01', 'matriarch.n.01', 'shiksa.n.01', 'widow.n.01', 'wife.n.01', 'broad.n.01', 'cinderella.n.01')
eat.v.01 -> ('gorge.v.01', 'sample.v.01', 'satiate.v.01', 'feast.v.01', 'cannibalize.v.01', 'drink.v.01', 'nosh.v.01', 'swallow.v.01', 'breakfast.v.01', 'brunch.v.01', 'dine.v.01', 'lunch.v.01', 'mess.v.01', 'picnic.v.01', 'smoke.v.01', 'sup.v.01', 'toast.v.02', 'drink.v.02', 'drug.v.02', 'use.v.02')
spicy.a.01 -> ('malodorous.a.01', 'palatable.a.01', 'aromatic.a.01', 'fragrant.a.01', 'hot.a.01', 'sweet.a.01', 'tasteless.a.01', 'tasty.a.01', 'unpalatable.a.01', 'salty.a.02', 'sour.a.02')
new.a.01 -> ('young.a.01', 'familiar.a.01', 'fresh.a.01', 'mature.a.01', 'old.a.01', 'unfamiliar.a.01', 'old.a.02')


In [28]:
# 被共享 lemma 规则挡掉的候选：池里有，但最终没进 candidates()
SID = "see.v.01"
ss = wn.synset(SID)
ref = {l.name().lower() for l in ss.lemmas()}
for s in sorted(wc._pool(ss), key=lambda s: s.name()):
    shared = {l.name().lower() for l in s.lemmas()} & ref
    if shared:
        print(s.name(), "shares", shared)

determine.v.08 shares {'see'}
examine.v.02 shares {'see'}
learn.v.02 shares {'see'}
see.v.11 shares {'see'}
see.v.12 shares {'see'}
see.v.19 shares {'see'}
visit.v.01 shares {'see'}
visualize.v.01 shares {'see'}
watch.v.03 shares {'see'}
witness.v.02 shares {'see'}
